# EasyRAG PDF And Multimodal Loading

## What you will do

- simulate repository PDF loading without relying on a real parser
- inspect per-page metadata, image extraction, and scanned-page placeholders
- trace how PDF-derived documents become multimodal embedding inputs

## Why this matters

PDF loading is the point where non-text sources have to become normal EasyRAG `Document` objects. If that conversion is unclear, later indexing and retrieval behavior feels arbitrary.


## Source code anchors

- `easyrag.rag.indexing.loaders.load_repo_documents`
- `easyrag.rag.indexing.loaders.load_pdf_documents`
- `easyrag.rag.indexing.loaders.build_document_metadata`
- `easyrag.rag.indexing.pipeline._build_embedding_input`


In [ ]:
# ruff: noqa: E402,F401,F811
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "easyrag").exists():
        REPO_ROOT = candidate.resolve()
        if str(REPO_ROOT) not in sys.path:
            sys.path.insert(0, str(REPO_ROOT))
        break
else:
    raise RuntimeError("Could not locate the EasyRAG repository root.")

import json
import os
import tempfile
import time
from pathlib import Path
from pprint import pprint
from unittest import mock

from easyrag.rag import AnswerParam, EasyRAG, EvalCase, QueryParam
from easyrag.support.async_utils import run_sync
from notebooks._utils import (
    managed_demo_rag,
    print_json as _print_json,
    production_backends_ready,
    provider_ready as can_use_openai_compatible_models,
    skip_message,
    stub_embedding as _stub_embedding,
    stub_query_model as _stub_query_model,
    stub_reranker as _stub_reranker,
)

from easyrag.rag.indexing.loaders import build_document_metadata, load_repo_documents, load_pdf_documents
from easyrag.rag.indexing.pipeline import build_insert_payloads


## Deterministic path

We use mocked PDF pages so the notebook stays stable on any machine. That lets us inspect the loader behavior directly instead of depending on a particular PDF parser installation.


In [ ]:
repo_tmp = tempfile.TemporaryDirectory()
repo_root = Path(repo_tmp.name) / "demo_repo"
docs_dir = repo_root / "docs"
docs_dir.mkdir(parents=True)
(docs_dir / "overview.md").write_text("# Overview\nEasyRAG keeps PDF and text inputs on one document contract.\n", encoding="utf-8")
pdf_path = docs_dir / "manual.pdf"
pdf_path.write_bytes(b"%PDF-1.4 fake")

fake_image = mock.Mock(name="diagram.png", data=b"PNGDATA")
fake_pages = [
    mock.Mock(extract_text=mock.Mock(return_value="Page one architecture notes"), images=[]),
    mock.Mock(extract_text=mock.Mock(return_value=""), images=[fake_image]),
    mock.Mock(extract_text=mock.Mock(return_value="Page three setup details"), images=[]),
]

with mock.patch("easyrag.rag.indexing.loaders.PdfReader", return_value=mock.Mock(pages=fake_pages)):
    pdf_documents = load_pdf_documents(pdf_path, repo_root)
    repo_documents = load_repo_documents(repo_root)

print("PDF page documents")
for document in pdf_documents:
    print(document.page_content)
    _print_json(document.metadata)
    print()

print("All repo documents")
for document in repo_documents:
    _print_json(document.metadata)


## Output inspection

The image-only page still becomes a `Document`. EasyRAG inserts a small scanned-page placeholder so the page is visible to later stages, and the metadata keeps the extracted image paths for multimodal embedding or rerank flows.


In [ ]:
pdf_only_rag = EasyRAG(
    working_dir=repo_tmp.name,
    workspace="pdf-loading-demo",
    embedding_func=_stub_embedding,
    query_model_func=_stub_query_model,
)
pdf_payloads = build_insert_payloads(pdf_only_rag, pdf_documents)

print("Document payload for page 2")
second_page = next(item for item in pdf_payloads["documents"] if item["metadata"].get("page_number") == 2)
_print_json(second_page)
print("\nVector chunk embedding inputs")
for item in pdf_payloads["vector_chunks"]:
    _print_json({
        "id": item["id"],
        "embedding_input": item["embedding_input"],
        "image_paths": item["metadata"].get("image_paths", []),
    })


## Provider-backed path

If OpenAI-compatible settings are available, the next cell shows what a provider-backed workspace does with the same PDF-derived documents. The point is to confirm that the multimodal metadata survives the handoff.


In [ ]:
if not can_use_openai_compatible_models():
    print("Skipping provider-backed path because OPENAI-compatible config is not set.")
else:
    provider_rag = EasyRAG(working_dir=repo_tmp.name, workspace="provider-pdf-loading-demo")
    try:
        provider_payloads = build_insert_payloads(provider_rag, pdf_documents)
        for item in provider_payloads["vector_chunks"]:
            _print_json({"id": item["id"], "embedding_input": item["embedding_input"]})
    except Exception as exc:
        print(f"Provider-backed path was skipped due to runtime error: {exc}")


## What changed and why

The important change is not that PDF pages suddenly become plain text. The important change is that every page now has a stable `doc_id`, `page_number`, source type, and optional image payload. That lets the rest of the pipeline treat PDFs as first-class indexed documents instead of a special case.


In [ ]:
repo_tmp.cleanup()
print("Cleaned up the temporary PDF-loading workspace.")


## Next steps

- Continue with [02_04_normalization_and_cleaning.ipynb](02_04_normalization_and_cleaning.ipynb) to see how extraction-heavy sources are normalized before indexing.
- Read [02-data-loading-overview.md](../../docs/02-data-loading-overview.md) for the chapter-level loading map.
- Revisit [00_02_observing_rag_artifacts.ipynb](../00_overview/00_02_observing_rag_artifacts.ipynb) if you want to compare PDF artifacts with the end-to-end lifecycle view.
